# 02 — Evaluate Agents with Built-in MLflow Judges

This notebook shows how to use MLflow 3.x **built-in scorers** to automatically
evaluate whether your agent calls tools correctly and responds appropriately.

You'll learn:
* What built-in scorers are (`ToolCallEfficiency`, `RelevanceToQuery`, `Safety`)
* How `mlflow.genai.evaluate()` runs your agent and applies judges to each trace
* How to define custom scorers with `Guidelines`
* How to interpret pass/fail results and justifications

**Compute:** DBR 18.2 ML cluster (`mlflow-eval-suite`)  
**Dependencies:** All pre-installed — no `%pip install` required.

**Prerequisites:** Run notebook 01 first to understand traces and autologging.

In [0]:
# No installs needed — DBR 18.2 ML ships with mlflow 3.x, openai, and databricks-agents.
import mlflow
print(f"MLflow {mlflow.__version__} — ready to go.")

## What are Built-in Scorers?

MLflow 3.x provides **built-in scorers** — pre-configured LLM evaluators that inspect the full
**trace** (not just the text output) to assess agent behavior.

Key scorers for agentic apps:
* `ToolCallEfficiency` — Did the agent avoid redundant/duplicate tool calls?
* `RelevanceToQuery` — Does the response address the user’s question?
* `Safety` — Is the response free of harmful content?
* `Correctness` — Does the output match ground truth expectations?
* `Guidelines` — Does the response follow custom rules you define?

**Important:** These scorers inspect `TOOL`-type spans in the trace. They work with agents
where tool execution is captured as a span (via `@mlflow.trace` or `mlflow.start_span`).

In [0]:
"""Setup: same agent as notebook 01 + import scorers."""
import openai
import json
from mlflow.entities import SpanType
from mlflow.genai.scorers import ToolCallEfficiency, RelevanceToQuery, Safety, Guidelines

# Enable OpenAI autologging
mlflow.openai.autolog()

# Experiment for this evaluation
_username = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
)
EXPERIMENT_NAME = f"/Users/{_username}/agentic-evals-intro"
mlflow.set_experiment(EXPERIMENT_NAME)

# --- Tool definition ---
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "lookup_order",
            "description": "Look up the status of a customer order by order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order ID (e.g. ORD-1001)"}
                },
                "required": ["order_id"],
            },
        },
    }
]

def lookup_order(order_id: str) -> str:
    """Look up the status of a customer order by order ID."""
    orders = {
        "ORD-1001": "Shipped \u2014 arrives Thursday",
        "ORD-1002": "Processing \u2014 payment confirmed, preparing for shipment",
        "ORD-1003": "Delivered \u2014 left at front door on Monday",
        "ORD-1004": "Cancelled \u2014 refund issued",
    }
    return orders.get(order_id, f"Order {order_id} not found in system")

# --- Agent ---
MODEL_ENDPOINT = "databricks-claude-sonnet-4"
TOOL_FUNCTIONS = {"lookup_order": lookup_order}

client = openai.OpenAI(
    base_url=(
        f"https://{dbutils.notebook.entry_point.getDbutils().notebook().getContext().browserHostName().get()}"
        "/serving-endpoints"
    ),
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
)


@mlflow.trace(span_type=SpanType.AGENT)
def run_agent(user_message: str) -> str:
    """Simple tool-calling agent: LLM → tool execution → LLM."""
    messages = [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model=MODEL_ENDPOINT, messages=messages, tools=TOOLS
    )
    assistant_msg = response.choices[0].message

    if assistant_msg.tool_calls:
        messages.append({
            "role": "assistant",
            "content": assistant_msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in assistant_msg.tool_calls
            ],
        })

        for tool_call in assistant_msg.tool_calls:
            fn_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)

            with mlflow.start_span(name=fn_name, span_type=SpanType.TOOL) as span:
                span.set_inputs(args)
                result = TOOL_FUNCTIONS[fn_name](**args)
                span.set_outputs({"result": result})

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

        response = client.chat.completions.create(model=MODEL_ENDPOINT, messages=messages)
        return response.choices[0].message.content

    return assistant_msg.content


print(f"\u2713 Agent ready (LLM: {MODEL_ENDPOINT}, Tools: [lookup_order])")
print(f"  Experiment: {EXPERIMENT_NAME}")

## Step 1: Define an Evaluation Dataset

We define test questions. The built-in scorers work in **ground-truth free** mode by default —
they assess tool usage and response quality based on the trace alone.

Optionally, we can provide `expected_tool_calls` for stricter evaluation.

In [0]:
"""Define test questions for evaluation."""
import pandas as pd

eval_data = pd.DataFrame([
    {
        "inputs": {"messages": [{"role": "user", "content": "What's the status of order ORD-1001?"}]},
    },
    {
        "inputs": {"messages": [{"role": "user", "content": "Can you check on ORD-1002 for me?"}]},
    },
    {
        "inputs": {"messages": [{"role": "user", "content": "Has order ORD-1003 been delivered yet?"}]},
    },
    {
        "inputs": {"messages": [{"role": "user", "content": "I want to know about order ORD-1004"}]},
    },
    {
        "inputs": {"messages": [{"role": "user", "content": "What does 'shipped' mean?"}]},
    },
    {
        "inputs": {"messages": [{"role": "user", "content": "How long does standard shipping usually take?"}]},
    },
])

print(f"Evaluation dataset: {len(eval_data)} questions")
print(f"  4 require tool calls (order lookups)")
print(f"  2 are general knowledge (no tool needed)")
display(eval_data)

## Step 2: Run Evaluation with Built-in Scorers

`mlflow.genai.evaluate()` will:
1. Run your agent on each question (via `predict_fn`), generating a trace with TOOL spans
2. Pass each trace to the built-in scorers
3. Each scorer inspects the trace and returns pass/fail with a justification

We use:
* `ToolCallEfficiency()` — Were there redundant/duplicate tool calls?
* `RelevanceToQuery()` — Does the response address the user’s question?
* `Safety()` — Is the response free of harmful content?

In [0]:
"""Run MLflow evaluation with built-in scorers."""


def predict_fn(messages: list) -> str:
    """Run the agent and return the final message content."""
    user_msg = messages[0]["content"]
    return run_agent(user_msg)


print("Running evaluation (calls the agent 6 times + judges each trace)...")
print("=" * 60)

eval_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        ToolCallEfficiency(),
        RelevanceToQuery(),
        Safety(),
    ],
)

print("\n\u2713 Evaluation complete!")
print(f"\nAggregate Metrics:")
for name, value in sorted(eval_results.metrics.items()):
    if isinstance(value, float):
        print(f"  {name}: {value:.3f}")
    else:
        print(f"  {name}: {value}")

## Step 3: Inspect Per-Question Results

Let's look at what the scorers said for each question \u2014 did it pass or fail, and why?

In [0]:
"""Inspect the per-question evaluation results."""

# Display the eval results DataFrame
print(f"Available tables: {list(eval_results.tables.keys()) if eval_results.tables else 'None'}")
print(f"\nMetrics summary:")
for name, value in sorted(eval_results.metrics.items()):
    print(f"  {name}: {value}")

# Display per-row results if available
if eval_results.tables:
    results_df = list(eval_results.tables.values())[0]
    display(results_df)

## How Built-in Scorers Work Under the Hood

The built-in scorers inspect the **MLflow trace** to find:
1. **Available tools** — what tools were defined for the agent
2. **Tool calls made** — which tools the agent actually invoked (from TOOL spans)
3. **Arguments used** — what arguments were passed to each tool
4. **User intent** — what the user was asking for

`ToolCallEfficiency` evaluates: *"Were there redundant or duplicate tool calls?"*  
`RelevanceToQuery` evaluates: *"Does the final response address the user’s question?"*  
`Safety` evaluates: *"Is the response free of harmful content?"*

This is why agents need to emit `TOOL` spans (via `mlflow.start_span`) — the scorers
need to see tool execution details inside the trace to assess tool usage quality.

## Bonus: Custom Guidelines Scorer

You can define custom evaluation rules with `Guidelines`. This lets you encode domain-specific
requirements as a scorer — for example, ensuring the agent only calls the tool for specific order IDs.

In [0]:
"""Run evaluation with a custom Guidelines scorer."""

# Define custom guidelines for tool usage appropriateness
tool_usage_guidelines = Guidelines(
    name="tool_usage_appropriateness",
    guidelines=(
        "The agent should ONLY call the lookup_order tool when the user asks about "
        "a specific order ID (e.g., ORD-1001, ORD-1002). For general questions about "
        "shipping, order terminology, or other topics, the agent should answer directly "
        "without calling any tools. Evaluate whether the agent made the correct decision "
        "about tool usage."
    ),
)

print("Running evaluation with custom Guidelines scorer...")
print("=" * 60)

eval_results_guidelines = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[tool_usage_guidelines],
)

print("\n\u2713 Custom Guidelines evaluation complete!")
print(f"\nMetrics:")
for name, value in sorted(eval_results_guidelines.metrics.items()):
    if isinstance(value, float):
        print(f"  {name}: {value:.3f}")
    else:
        print(f"  {name}: {value}")

## Summary

What you learned:
* `ToolCallEfficiency` evaluates whether the agent avoids redundant/duplicate tool calls
* `RelevanceToQuery` checks if the response addresses the user’s question
* `Safety` detects harmful content in responses
* `Guidelines` lets you define custom evaluation rules (like tool usage appropriateness)
* `mlflow.genai.evaluate()` runs your agent on a dataset and applies scorers to each trace
* Results include pass/fail per question plus LLM-generated justifications

**Next:** In notebook 03, we build **custom judges** — code-based scorers (`@scorer`) and
LLM judges (`make_judge()`) for domain-specific evaluation rules.